In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dansbecker/melbourne-housing-snapshot/melb_data.csv


# Basic Machine Learning Practice with Melbourne Housing Snapshot

In this notebook, I practice a basic machine learning workflow using the Melbourne Housing Snapshot dataset. I load the data, choose features, train models, make predictions, and evaluate model performance using Mean Absolute Error.

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


## Load the Data

In [8]:
melbourne_file_path = "/kaggle/input/datasets/dansbecker/melbourne-housing-snapshot/melb_data.csv"

melbourne_data = pd.read_csv(melbourne_file_path)

melbourne_data.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


## Explore the Data

In [14]:
melbourne_data.shape

(13580, 21)

In [15]:
melbourne_data.columns


Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='object')

In [13]:
melbourne_data.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


## Clean the Data

In [16]:
melbourne_data = melbourne_data.dropna(axis=0)

melbourne_data.shape

(6196, 21)

## Choose Target and Features

In [18]:
y = melbourne_data.Price

melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 
                      'BuildingArea', 'YearBuilt', 
                      'Lattitude', 'Longtitude']

X = melbourne_data[melbourne_features]

X.head()

,Rooms,Bathroom,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude
1,2,1.0,156.0,79.0,1900.0,-37.8079,144.9934
2,3,2.0,134.0,150.0,1900.0,-37.8093,144.9944
4,4,1.0,120.0,142.0,2014.0,-37.8072,144.9941
6,3,2.0,245.0,210.0,1910.0,-37.8024,144.9993
7,2,1.0,256.0,107.0,1890.0,-37.8060,144.9954


## Split the Data

In [19]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

## Train a Decision Tree Model

In [20]:
tree_model = DecisionTreeRegressor(random_state=1)

tree_model.fit(train_X, train_y)

tree_predictions = tree_model.predict(val_X)

tree_mae = mean_absolute_error(val_y, tree_predictions)

print("Decision Tree MAE:", tree_mae)

Decision Tree MAE: 251876.65138799226


## Compare Different Decision Tree Sizes

In [21]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    predictions = model.predict(val_X)
    mae = mean_absolute_error(val_y, predictions)
    return mae

candidate_leaf_nodes = [5, 25, 50, 100, 250, 500]

for leaf_size in candidate_leaf_nodes:
    mae = get_mae(leaf_size, train_X, val_X, train_y, val_y)
    print("Max leaf nodes:", leaf_size, "MAE:", mae)

Max leaf nodes: 5 MAE: 324110.91454760684
Max leaf nodes: 25 MAE: 263595.3211945369
Max leaf nodes: 50 MAE: 248796.2586659669
Max leaf nodes: 100 MAE: 238658.40293237285
Max leaf nodes: 250 MAE: 236125.32714056323
Max leaf nodes: 500 MAE: 239296.12571612737


In [22]:
scores = {
    leaf_size: get_mae(leaf_size, train_X, val_X, train_y, val_y)
    for leaf_size in candidate_leaf_nodes
}

best_tree_size = min(scores, key=scores.get)

print("Best tree size:", best_tree_size)

Best tree size: 250


## Train a Random Forest Model

In [23]:
forest_model = RandomForestRegressor(random_state=1)

forest_model.fit(train_X, train_y)

forest_predictions = forest_model.predict(val_X)

forest_mae = mean_absolute_error(val_y, forest_predictions)

print("Random Forest MAE:", forest_mae)

Random Forest MAE: 173864.25945341078


## Compare Model Results

In [24]:
print("Decision Tree MAE:", tree_mae)
print("Random Forest MAE:", forest_mae)

Decision Tree MAE: 251876.65138799226
Random Forest MAE: 173864.25945341078


The Random Forest model performed better than the basic Decision Tree model because it had a lower MAE. This means the Random Forest predictions were closer to the actual house prices on average.